# tsfresh Feature Filter
Filter features before passing to tsfresh based on 10 exclusion rules + autocorrelation gate:
- ❌ Near-constant, white-noise, monotonic, sparse, smoothed, lag, redundant features
- ❌ Short time series, indexing artifacts, target leakage
- ✅ Autocorrelation gate: 0.05 < |lag-1| AND 0.03 < |lag-2| (exclude if too high: >0.95, >0.90)

In [1]:
import polars as pl
import numpy as np
from pathlib import Path
from typing import Dict, List, Set, Tuple
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../../data/raw")
COMBINED_PATH = DATA_DIR / "combined/train.parquet"

In [2]:
# Load data
df = pl.read_parquet(COMBINED_PATH)
print(f"Shape: {df.shape}")

# Define column groups
ID_COLS = ["id", "code", "sub_code", "sub_category"]
META_COLS = ["horizon", "ts_index"]
TARGET_COLS = ["y_target", "weight"]
FEATURE_COLS = [c for c in df.columns if c.startswith("feature_")]

print(f"Features: {len(FEATURE_COLS)}")

Shape: (5337414, 94)
Features: 86


## Feature Exclusion Rules Configuration

In [3]:
# Thresholds for exclusion rules
CONFIG = {
    "near_constant_std_threshold": 1e-6,       # Rule 1: std ≈ 0
    "white_noise_autocorr_threshold": 0.05,   # Rule 2: |lag-1 autocorr| ≈ 0
    "monotonic_corr_threshold": 0.95,         # Rule 3: |corr(feature, time)| ≈ 1
    "sparse_null_ratio_threshold": 0.50,      # Rule 4: non-null ratio < 50%
    "smoothed_rolling_corr_threshold": 0.98,  # Rule 5: high corr with rolling mean
    "rolling_window_size": 5,                 # Rule 5: window for rolling mean
    "redundant_corr_threshold": 0.95,         # Rule 7: |corr(fi, fj)| > 0.95
    "min_series_length": 10,                  # Rule 8: min points per entity
    "drop_series_length": 5,                  # Rule 8: absolute drop threshold
    "leakage_corr_threshold": 0.90,           # Rule 10: suspiciously high target corr
}

# Autocorrelation gate thresholds (Rule 11)
# PASS if: lag1_min < |lag-1| < lag1_max AND lag2_min < |lag-2| < lag2_max
AUTOCORR_GATE = {
    "lag1_min": 0.05,  # |lag-1 autocorr| > 0.05 (exclude too low)
    "lag2_min": 0.03,  # |lag-2 autocorr| > 0.03 (exclude too low)
    "lag1_max": 0.95,  # |lag-1 autocorr| < 0.95 (exclude too high)
    "lag2_max": 0.90,  # |lag-2 autocorr| < 0.90 (exclude too high)
}

## Rule 1: Near-Constant Features
Exclude features with very low variance (std ≈ 0)

In [4]:
def check_near_constant(df: pl.DataFrame, features: List[str], threshold: float) -> Set[str]:
    """Rule 1: Find features with std ≈ 0 (near-constant)"""
    excluded = set()
    
    # Group by entity (sub_code) and compute std over time
    for feat in features:
        entity_stds = (
            df.group_by("sub_code")
            .agg(pl.col(feat).std().alias("std"))
            .select("std")
            .to_series()
            .drop_nulls()
        )
        
        # If median std across entities is near zero, exclude
        if len(entity_stds) > 0 and entity_stds.median() < threshold:
            excluded.add(feat)
    
    return excluded

near_constant = check_near_constant(df, FEATURE_COLS, CONFIG["near_constant_std_threshold"])
print(f"❌ Rule 1 (Near-constant): {len(near_constant)} features")
if near_constant:
    print(f"   {sorted(near_constant)[:10]}..." if len(near_constant) > 10 else f"   {sorted(near_constant)}")

❌ Rule 1 (Near-constant): 0 features


## Rule 2: White-Noise Features
Exclude features with |lag-1 autocorrelation| ≈ 0

In [5]:
def check_white_noise(df: pl.DataFrame, features: List[str], threshold: float) -> Set[str]:
    """Rule 2: Find features with no temporal dependence (|lag-1 autocorr| ≈ 0)"""
    excluded = set()
    
    # Sample entities for efficiency
    sample_entities = df.select("sub_code").unique().sample(n=min(50, df.select("sub_code").n_unique()))
    df_sample = df.filter(pl.col("sub_code").is_in(sample_entities.to_series()))
    
    for feat in features:
        autocorrs = []
        
        for entity in sample_entities.to_series():
            entity_data = (
                df_sample.filter(pl.col("sub_code") == entity)
                .sort("ts_index")
                .select(feat)
                .to_series()
                .drop_nulls()
            )
            
            if len(entity_data) > 2:
                # Compute lag-1 autocorrelation
                x = entity_data.to_numpy()
                if np.std(x) > 1e-10:
                    autocorr = np.corrcoef(x[:-1], x[1:])[0, 1]
                    if not np.isnan(autocorr):
                        autocorrs.append(abs(autocorr))
        
        # If median autocorrelation is very low, it's white noise
        if autocorrs and np.median(autocorrs) < threshold:
            excluded.add(feat)
    
    return excluded

white_noise = check_white_noise(df, FEATURE_COLS, CONFIG["white_noise_autocorr_threshold"])
print(f"❌ Rule 2 (White-noise): {len(white_noise)} features")
if white_noise:
    print(f"   {sorted(white_noise)[:10]}..." if len(white_noise) > 10 else f"   {sorted(white_noise)}")

❌ Rule 2 (White-noise): 6 features
   ['feature_b', 'feature_c', 'feature_d', 'feature_e', 'feature_f', 'feature_g']


## Rule 3: Monotonic / Counter-like Features
Exclude features with |corr(feature, time_index)| ≈ 1

In [6]:
def check_monotonic(df: pl.DataFrame, features: List[str], threshold: float) -> Set[str]:
    """Rule 3: Find purely monotonic/counter features with |corr(feat, time)| ≈ 1"""
    excluded = set()
    
    sample_entities = df.select("sub_code").unique().sample(n=min(50, df.select("sub_code").n_unique()))
    df_sample = df.filter(pl.col("sub_code").is_in(sample_entities.to_series()))
    
    for feat in features:
        corrs = []
        
        for entity in sample_entities.to_series():
            entity_data = (
                df_sample.filter(pl.col("sub_code") == entity)
                .sort("ts_index")
                .select(["ts_index", feat])
                .drop_nulls()
            )
            
            if len(entity_data) > 5:
                time_idx = entity_data["ts_index"].to_numpy().astype(float)
                feat_vals = entity_data[feat].to_numpy()
                
                if np.std(feat_vals) > 1e-10 and np.std(time_idx) > 1e-10:
                    corr = abs(np.corrcoef(time_idx, feat_vals)[0, 1])
                    if not np.isnan(corr):
                        corrs.append(corr)
        
        if corrs and np.median(corrs) > threshold:
            excluded.add(feat)
    
    return excluded

monotonic = check_monotonic(df, FEATURE_COLS, CONFIG["monotonic_corr_threshold"])
print(f"❌ Rule 3 (Monotonic): {len(monotonic)} features")
if monotonic:
    print(f"   {sorted(monotonic)[:10]}..." if len(monotonic) > 10 else f"   {sorted(monotonic)}")

❌ Rule 3 (Monotonic): 1 features
   ['feature_a']


## Rule 4: Sparse Features
Exclude features with non-null ratio < 50%

In [7]:
def check_sparse(df: pl.DataFrame, features: List[str], threshold: float) -> Set[str]:
    """Rule 4: Find features with non-null ratio < threshold"""
    excluded = set()
    total_rows = len(df)
    
    for feat in features:
        non_null_count = df.select(pl.col(feat).is_not_null().sum()).item()
        non_null_ratio = non_null_count / total_rows
        
        if non_null_ratio < threshold:
            excluded.add(feat)
    
    return excluded

sparse = check_sparse(df, FEATURE_COLS, CONFIG["sparse_null_ratio_threshold"])
print(f"❌ Rule 4 (Sparse): {len(sparse)} features")
if sparse:
    print(f"   {sorted(sparse)[:10]}..." if len(sparse) > 10 else f"   {sorted(sparse)}")

❌ Rule 4 (Sparse): 0 features


## Rule 5: Already Aggregated / Smoothed Features
Exclude features highly correlated with their own rolling mean

In [8]:
def check_smoothed(df: pl.DataFrame, features: List[str], 
                   threshold: float, window: int) -> Set[str]:
    """Rule 5: Find already smoothed features (high corr with rolling mean)"""
    excluded = set()
    
    sample_entities = df.select("sub_code").unique().sample(n=min(30, df.select("sub_code").n_unique()))
    df_sample = df.filter(pl.col("sub_code").is_in(sample_entities.to_series()))
    
    for feat in features:
        corrs = []
        
        for entity in sample_entities.to_series():
            entity_data = (
                df_sample.filter(pl.col("sub_code") == entity)
                .sort("ts_index")
                .select(feat)
                .drop_nulls()
            )
            
            if len(entity_data) > window + 2:
                values = entity_data.to_series().to_numpy()
                
                # Compute rolling mean
                rolling_mean = np.convolve(values, np.ones(window)/window, mode='valid')
                original = values[window-1:]
                
                if len(original) == len(rolling_mean) and np.std(original) > 1e-10:
                    corr = np.corrcoef(original, rolling_mean)[0, 1]
                    if not np.isnan(corr):
                        corrs.append(abs(corr))
        
        if corrs and np.median(corrs) > threshold:
            excluded.add(feat)
    
    return excluded

smoothed = check_smoothed(df, FEATURE_COLS, CONFIG["smoothed_rolling_corr_threshold"], 
                          CONFIG["rolling_window_size"])
print(f"❌ Rule 5 (Smoothed): {len(smoothed)} features")
if smoothed:
    print(f"   {sorted(smoothed)[:10]}..." if len(smoothed) > 10 else f"   {sorted(smoothed)}")

❌ Rule 5 (Smoothed): 1 features
   ['feature_a']


## Rule 6: Pure Lag / Snapshot Features
Identify features that represent single past values (pattern analysis)

In [9]:
def check_lag_features(df: pl.DataFrame, features: List[str]) -> Set[str]:
    """Rule 6: Identify lag/snapshot features by checking if values repeat in shifted pattern"""
    excluded = set()
    
    sample_entities = df.select("sub_code").unique().sample(n=min(20, df.select("sub_code").n_unique()))
    df_sample = df.filter(pl.col("sub_code").is_in(sample_entities.to_series()))
    
    for feat in features:
        is_lag = []
        
        for entity in sample_entities.to_series():
            entity_data = (
                df_sample.filter(pl.col("sub_code") == entity)
                .sort("ts_index")
                .select(feat)
                .drop_nulls()
                .to_series()
            )
            
            if len(entity_data) > 5:
                values = entity_data.to_numpy()
                
                # Check for lag-1 pattern: v[t] appears at v[t+1]
                for lag in [1, 2, 3]:
                    if len(values) > lag:
                        # Perfect lag if values match exactly when shifted
                        if np.allclose(values[:-lag], values[lag:], rtol=1e-8, equal_nan=True):
                            is_lag.append(True)
                            break
                else:
                    is_lag.append(False)
        
        # If most entities show lag pattern, exclude
        if is_lag and sum(is_lag) / len(is_lag) > 0.5:
            excluded.add(feat)
    
    return excluded

lag_features = check_lag_features(df, FEATURE_COLS)
print(f"❌ Rule 6 (Lag/Snapshot): {len(lag_features)} features")
if lag_features:
    print(f"   {sorted(lag_features)[:10]}..." if len(lag_features) > 10 else f"   {sorted(lag_features)}")

❌ Rule 6 (Lag/Snapshot): 0 features


## Rule 7: Redundant / Duplicate Features
Exclude features with |corr(fi, fj)| > 0.95

In [10]:
def check_redundant(df: pl.DataFrame, features: List[str], threshold: float) -> Set[str]:
    """Rule 7: Find highly correlated feature pairs and keep only one"""
    excluded = set()
    
    # Sample data for efficiency
    df_sample = df.sample(n=min(50000, len(df)))
    
    # Get numeric values
    feature_data = df_sample.select(features).to_pandas()
    
    # Compute correlation matrix
    corr_matrix = feature_data.corr().abs()
    
    # Find highly correlated pairs
    for i, feat_i in enumerate(features):
        if feat_i in excluded:
            continue
        for feat_j in features[i+1:]:
            if feat_j in excluded:
                continue
            
            corr_val = corr_matrix.loc[feat_i, feat_j]
            if not np.isnan(corr_val) and corr_val > threshold:
                # Keep feat_i, exclude feat_j
                excluded.add(feat_j)
    
    return excluded

redundant = check_redundant(df, FEATURE_COLS, CONFIG["redundant_corr_threshold"])
print(f"❌ Rule 7 (Redundant): {len(redundant)} features")
if redundant:
    print(f"   {sorted(redundant)[:10]}..." if len(redundant) > 10 else f"   {sorted(redundant)}")

❌ Rule 7 (Redundant): 3 features
   ['feature_af', 'feature_bo', 'feature_cd']


## Rule 8: Short Time Series Check
Flag entities with < 10 points (risky), drop if < 5 points

In [11]:
def check_short_series(df: pl.DataFrame, min_length: int, drop_length: int) -> Tuple[List[str], List[str]]:
    """Rule 8: Find entities with short time series"""
    
    series_lengths = (
        df.group_by("sub_code")
        .agg(pl.len().alias("length"))
    )
    
    risky = series_lengths.filter(pl.col("length") < min_length).select("sub_code").to_series().to_list()
    drop = series_lengths.filter(pl.col("length") < drop_length).select("sub_code").to_series().to_list()
    
    return risky, drop

risky_entities, drop_entities = check_short_series(df, CONFIG["min_series_length"], CONFIG["drop_series_length"])
print(f"⚠️  Rule 8 (Short series): {len(risky_entities)} risky entities (<{CONFIG['min_series_length']} points)")
print(f"❌ Rule 8 (Short series): {len(drop_entities)} entities to drop (<{CONFIG['drop_series_length']} points)")

⚠️  Rule 8 (Short series): 0 risky entities (<10 points)
❌ Rule 8 (Short series): 0 entities to drop (<5 points)


## Rule 9: Construction / Indexing Artifacts
Detect features that are deterministic by design (encode position/count)

In [12]:
def check_indexing_artifacts(df: pl.DataFrame, features: List[str]) -> Set[str]:
    """Rule 9: Find features that are indexing/counting artifacts"""
    excluded = set()
    
    sample_entities = df.select("sub_code").unique().sample(n=min(30, df.select("sub_code").n_unique()))
    df_sample = df.filter(pl.col("sub_code").is_in(sample_entities.to_series()))
    
    for feat in features:
        is_artifact = []
        
        for entity in sample_entities.to_series():
            entity_data = (
                df_sample.filter(pl.col("sub_code") == entity)
                .sort("ts_index")
                .select(["ts_index", feat])
                .drop_nulls()
            )
            
            if len(entity_data) > 3:
                feat_vals = entity_data[feat].to_numpy()
                ts_idx = entity_data["ts_index"].to_numpy()
                
                # Check if feature equals ts_index (position encoding)
                if np.allclose(feat_vals, ts_idx, rtol=1e-8):
                    is_artifact.append(True)
                    continue
                
                # Check if feature is row count (cumsum pattern)
                row_positions = np.arange(len(feat_vals))
                if np.allclose(feat_vals, row_positions, rtol=1e-8):
                    is_artifact.append(True)
                    continue
                
                # Check for constant integer differences (counter)
                if len(feat_vals) > 1:
                    diffs = np.diff(feat_vals)
                    if np.std(diffs) < 1e-10 and len(np.unique(diffs)) == 1:
                        is_artifact.append(True)
                        continue
                
                is_artifact.append(False)
        
        if is_artifact and sum(is_artifact) / len(is_artifact) > 0.5:
            excluded.add(feat)
    
    return excluded

artifacts = check_indexing_artifacts(df, FEATURE_COLS)
print(f"❌ Rule 9 (Indexing artifacts): {len(artifacts)} features")
if artifacts:
    print(f"   {sorted(artifacts)[:10]}..." if len(artifacts) > 10 else f"   {sorted(artifacts)}")

❌ Rule 9 (Indexing artifacts): 0 features


## Rule 10: Target Leakage Detection
Flag features with suspiciously high correlation to target

In [13]:
def check_target_leakage(df: pl.DataFrame, features: List[str], threshold: float) -> Set[str]:
    """Rule 10: Find features with suspiciously high target correlation (leakage)"""
    excluded = set()
    
    df_sample = df.sample(n=min(100000, len(df))).drop_nulls(subset=["y_target"])
    
    target = df_sample["y_target"].to_numpy()
    
    for feat in features:
        feat_data = df_sample[feat].to_numpy()
        
        # Mask nulls
        mask = ~np.isnan(feat_data) & ~np.isnan(target)
        
        if mask.sum() > 100:
            corr = abs(np.corrcoef(feat_data[mask], target[mask])[0, 1])
            
            if not np.isnan(corr) and corr > threshold:
                excluded.add(feat)
                print(f"   ⚠️  {feat}: corr = {corr:.4f} (suspicious!)")
    
    return excluded

leakage = check_target_leakage(df, FEATURE_COLS, CONFIG["leakage_corr_threshold"])
print(f"❌ Rule 10 (Target leakage): {len(leakage)} features")

❌ Rule 10 (Target leakage): 0 features


## Summary: Combine All Exclusions (Rules 1-10)

In [14]:
# Combine all excluded features
all_excluded = (
    near_constant | white_noise | monotonic | sparse | 
    smoothed | lag_features | redundant | artifacts | leakage
)

# Features that passed exclusion rules (before autocorrelation gate)
tsfresh_features = [f for f in FEATURE_COLS if f not in all_excluded]

print("=" * 60)
print("EXCLUSION RULES SUMMARY (Rules 1-10)")
print("=" * 60)
print(f"\nTotal features analyzed: {len(FEATURE_COLS)}")
print(f"\n❌ EXCLUDED:")
print(f"   Rule 1 (Near-constant):    {len(near_constant):3d}")
print(f"   Rule 2 (White-noise):      {len(white_noise):3d}")
print(f"   Rule 3 (Monotonic):        {len(monotonic):3d}")
print(f"   Rule 4 (Sparse):           {len(sparse):3d}")
print(f"   Rule 5 (Smoothed):         {len(smoothed):3d}")
print(f"   Rule 6 (Lag/Snapshot):     {len(lag_features):3d}")
print(f"   Rule 7 (Redundant):        {len(redundant):3d}")
print(f"   Rule 9 (Index artifacts):  {len(artifacts):3d}")
print(f"   Rule 10 (Target leakage):  {len(leakage):3d}")
print(f"   --------------------------------")
print(f"   Total excluded (union):    {len(all_excluded):3d}")
print(f"\n✅ After exclusion rules: {len(tsfresh_features)} features")

EXCLUSION RULES SUMMARY (Rules 1-10)

Total features analyzed: 86

❌ EXCLUDED:
   Rule 1 (Near-constant):      0
   Rule 2 (White-noise):        6
   Rule 3 (Monotonic):          1
   Rule 4 (Sparse):             0
   Rule 5 (Smoothed):           1
   Rule 6 (Lag/Snapshot):       0
   Rule 7 (Redundant):          3
   Rule 9 (Index artifacts):    0
   Rule 10 (Target leakage):    0
   --------------------------------
   Total excluded (union):     10

✅ After exclusion rules: 76 features


In [15]:
# Exclusion breakdown by feature
exclusion_reasons = {}
for feat in all_excluded:
    reasons = []
    if feat in near_constant: reasons.append("near_constant")
    if feat in white_noise: reasons.append("white_noise")
    if feat in monotonic: reasons.append("monotonic")
    if feat in sparse: reasons.append("sparse")
    if feat in smoothed: reasons.append("smoothed")
    if feat in lag_features: reasons.append("lag_feature")
    if feat in redundant: reasons.append("redundant")
    if feat in artifacts: reasons.append("artifact")
    if feat in leakage: reasons.append("leakage")
    exclusion_reasons[feat] = reasons

print("\nExcluded features with reasons:")
for feat, reasons in sorted(exclusion_reasons.items()):
    print(f"  {feat}: {', '.join(reasons)}")


Excluded features with reasons:
  feature_a: monotonic, smoothed
  feature_af: redundant
  feature_b: white_noise
  feature_bo: redundant
  feature_c: white_noise
  feature_cd: redundant
  feature_d: white_noise
  feature_e: white_noise
  feature_f: white_noise
  feature_g: white_noise


## Rule 11: Autocorrelation Gate (Final Filter)
Keep ONLY features satisfying BOTH:
- |lag-1 autocorr| > 0.05 AND < 0.95
- |lag-2 autocorr| > 0.03 AND < 0.90

Exclude if too low (no signal) OR too high (over-smoothed)

In [16]:
def compute_autocorrelations(df: pl.DataFrame, features: List[str], 
                              lag1_min: float, lag2_min: float,
                              lag1_max: float, lag2_max: float) -> Tuple[Set[str], Set[str], Set[str], Dict]:
    """Rule 11: Keep only features with autocorrelation in acceptable range
    
    Returns:
        passed: Features that passed all conditions
        failed_too_low: Features excluded for too low autocorr
        failed_too_high: Features excluded for too high autocorr
        autocorr_stats: Detailed stats per feature
    """
    passed = set()
    failed_too_low = set()
    failed_too_high = set()
    autocorr_stats = {}
    
    # Sample entities for efficiency
    sample_entities = df.select("sub_code").unique().sample(n=min(50, df.select("sub_code").n_unique()))
    df_sample = df.filter(pl.col("sub_code").is_in(sample_entities.to_series()))
    
    for feat in features:
        lag1_corrs = []
        lag2_corrs = []
        
        for entity in sample_entities.to_series():
            entity_data = (
                df_sample.filter(pl.col("sub_code") == entity)
                .sort("ts_index")
                .select(feat)
                .to_series()
                .drop_nulls()
            )
            
            if len(entity_data) > 3:
                x = entity_data.to_numpy()
                
                if np.std(x) > 1e-10:
                    # Lag-1 autocorrelation
                    if len(x) > 1:
                        corr1 = np.corrcoef(x[:-1], x[1:])[0, 1]
                        if not np.isnan(corr1):
                            lag1_corrs.append(abs(corr1))
                    
                    # Lag-2 autocorrelation
                    if len(x) > 2:
                        corr2 = np.corrcoef(x[:-2], x[2:])[0, 1]
                        if not np.isnan(corr2):
                            lag2_corrs.append(abs(corr2))
        
        # Compute median autocorrelations
        median_lag1 = np.median(lag1_corrs) if lag1_corrs else 0
        median_lag2 = np.median(lag2_corrs) if lag2_corrs else 0
        
        # Check conditions
        passes_lag1_min = median_lag1 > lag1_min
        passes_lag2_min = median_lag2 > lag2_min
        passes_lag1_max = median_lag1 <= lag1_max
        passes_lag2_max = median_lag2 <= lag2_max
        
        # Too high check: BOTH lag1 > 0.95 AND lag2 > 0.90
        is_too_high = (median_lag1 > lag1_max) and (median_lag2 > lag2_max)
        
        autocorr_stats[feat] = {
            "lag1_autocorr": float(median_lag1),
            "lag2_autocorr": float(median_lag2),
            "passes_lag1_min": passes_lag1_min,
            "passes_lag2_min": passes_lag2_min,
            "passes_lag1_max": passes_lag1_max,
            "passes_lag2_max": passes_lag2_max,
            "is_too_high": is_too_high,
        }
        
        # Categorize: pass if meets min thresholds AND not too high
        if is_too_high:
            failed_too_high.add(feat)
        elif not (passes_lag1_min and passes_lag2_min):
            failed_too_low.add(feat)
        else:
            passed.add(feat)
    
    return passed, failed_too_low, failed_too_high, autocorr_stats

# Apply autocorrelation gate to features that passed exclusion rules
passed_autocorr_gate, failed_too_low, failed_too_high, autocorr_stats = compute_autocorrelations(
    df, tsfresh_features, 
    AUTOCORR_GATE["lag1_min"], AUTOCORR_GATE["lag2_min"],
    AUTOCORR_GATE["lag1_max"], AUTOCORR_GATE["lag2_max"]
)

# Features that failed the autocorrelation gate
failed_autocorr_gate = failed_too_low | failed_too_high

print(f"✅ Rule 11 (Autocorr Gate): {len(passed_autocorr_gate)} features PASSED")
print(f"❌ Rule 11 (Too low autocorr): {len(failed_too_low)} features")
print(f"❌ Rule 11 (Too high autocorr): {len(failed_too_high)} features")
if failed_too_high:
    print(f"   Too high (lag1>{AUTOCORR_GATE['lag1_max']} AND lag2>{AUTOCORR_GATE['lag2_max']}): {sorted(failed_too_high)}")

✅ Rule 11 (Autocorr Gate): 68 features PASSED
❌ Rule 11 (Too low autocorr): 0 features
❌ Rule 11 (Too high autocorr): 8 features
   Too high (lag1>0.95 AND lag2>0.9): ['feature_am', 'feature_az', 'feature_ce', 'feature_ch', 'feature_w', 'feature_x', 'feature_y', 'feature_z']


In [17]:
# Show autocorrelation stats for failed features
if failed_autocorr_gate:
    print("\nAutocorrelation details for FAILED features:")
    print(f"{'Feature':<20} {'Lag-1':>10} {'Lag-2':>10} {'Reason':>15}")
    print("-" * 60)
    for feat in sorted(failed_autocorr_gate):
        stats = autocorr_stats[feat]
        reason = "TOO HIGH" if stats['is_too_high'] else "TOO LOW"
        print(f"{feat:<20} {stats['lag1_autocorr']:>10.4f} {stats['lag2_autocorr']:>10.4f} {reason:>15}")


Autocorrelation details for FAILED features:
Feature                   Lag-1      Lag-2          Reason
------------------------------------------------------------
feature_am               0.9502     0.9003        TOO HIGH
feature_az               0.9683     0.9370        TOO HIGH
feature_ce               0.9510     0.9022        TOO HIGH
feature_ch               0.9772     0.9544        TOO HIGH
feature_w                0.9776     0.9555        TOO HIGH
feature_x                0.9616     0.9232        TOO HIGH
feature_y                0.9690     0.9380        TOO HIGH
feature_z                0.9722     0.9444        TOO HIGH


In [18]:
# Final feature list after all gates
tsfresh_features_final = sorted(list(passed_autocorr_gate))

print("=" * 60)
print("FINAL FEATURE LIST (After All Gates)")
print("=" * 60)
print(f"\nStarted with: {len(FEATURE_COLS)} features")
print(f"After exclusion rules (1-10): {len(tsfresh_features)} features")
print(f"After autocorrelation gate (Rule 11): {len(tsfresh_features_final)} features")
print(f"   - Removed (too low): {len(failed_too_low)}")
print(f"   - Removed (too high): {len(failed_too_high)}")
print(f"\n✅ FINAL tsfresh features:")
print(f"   {tsfresh_features_final[:15]}..." if len(tsfresh_features_final) > 15 else f"   {tsfresh_features_final}")

FINAL FEATURE LIST (After All Gates)

Started with: 86 features
After exclusion rules (1-10): 76 features
After autocorrelation gate (Rule 11): 68 features
   - Removed (too low): 0
   - Removed (too high): 8

✅ FINAL tsfresh features:
   ['feature_aa', 'feature_ab', 'feature_ac', 'feature_ad', 'feature_ae', 'feature_ag', 'feature_ah', 'feature_ai', 'feature_aj', 'feature_ak', 'feature_al', 'feature_an', 'feature_ao', 'feature_ap', 'feature_aq']...


## Export Filtered Feature List

In [20]:
# Save results
import json
import numpy as np

def to_json_safe(obj):
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    elif isinstance(obj, set):
        return [to_json_safe(v) for v in obj]
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    else:
        return obj

output_dir = Path("../../data/processed")
output_dir.mkdir(exist_ok=True, parents=True)

results = {
    "config": CONFIG,
    "autocorr_gate": AUTOCORR_GATE,
    "total_features": len(FEATURE_COLS),
    "excluded_features": sorted(list(all_excluded)),
    "after_exclusion_rules": tsfresh_features,
    "failed_autocorr_too_low": sorted(list(failed_too_low)),
    "failed_autocorr_too_high": sorted(list(failed_too_high)),
    "tsfresh_features_final": tsfresh_features_final,
    "autocorr_stats": autocorr_stats,
    "exclusion_reasons": exclusion_reasons,
    "risky_entities": risky_entities,
    "drop_entities": drop_entities,
    "rule_counts": {
        "near_constant": len(near_constant),
        "white_noise": len(white_noise),
        "monotonic": len(monotonic),
        "sparse": len(sparse),
        "smoothed": len(smoothed),
        "lag_features": len(lag_features),
        "redundant": len(redundant),
        "artifacts": len(artifacts),
        "leakage": len(leakage),
        "autocorr_too_low": len(failed_too_low),
        "autocorr_too_high": len(failed_too_high),
    }
}

results_json = to_json_safe(results)

with open(output_dir / "tsfresh_feature_filter_results.json", "w") as f:
    json.dump(results_json, f, indent=2)

print(f"\n✅ Results saved to {output_dir / 'tsfresh_feature_filter_results.json'}")


✅ Results saved to ../../data/processed/tsfresh_feature_filter_results.json


## Filter DataFrame for tsfresh

In [21]:
# Create filtered dataframe for tsfresh using FINAL features
# Keep: ID cols, time index, and passed features
tsfresh_cols = ["sub_code", "ts_index"] + tsfresh_features_final

# Filter out short series entities
df_tsfresh = (
    df.filter(~pl.col("sub_code").is_in(drop_entities))
    .select(tsfresh_cols)
)

print(f"tsfresh DataFrame shape: {df_tsfresh.shape}")
print(f"Columns: {df_tsfresh.columns[:10]}...")

# Save filtered data
df_tsfresh.write_parquet(output_dir / "tsfresh_filtered_data.parquet")
print(f"\n✅ Filtered data saved to {output_dir / 'tsfresh_filtered_data.parquet'}")

tsfresh DataFrame shape: (5337414, 70)
Columns: ['sub_code', 'ts_index', 'feature_aa', 'feature_ab', 'feature_ac', 'feature_ad', 'feature_ae', 'feature_ag', 'feature_ah', 'feature_ai']...

✅ Filtered data saved to ../../data/processed/tsfresh_filtered_data.parquet
